# Corpus Visualizer — mod/restitutiva
**Tenant:** `scriptorium` · **Database:** `mod-restitutiva`

Proyecta los embeddings de las colecciones Chroma de `mod/restitutiva` en 2D/3D usando UMAP.

**Requisitos:** `pip install chromadb umap-learn plotly pandas numpy scikit-learn`

In [6]:
%pip install -q chromadb umap-learn plotly pandas numpy scikit-learn "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.


In [7]:
import chromadb
import numpy as np
import pandas as pd
import umap
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from sklearn.preprocessing import LabelEncoder

def show(fig):
    """Renderiza una figura Plotly en VS Code sin depender de nbformat."""
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

# ── Conexión al cliente Chroma local persistente ────────────────────────────
STORAGE_PATH = r"C:\Users\aleph\OASIS\aleph-scriptorium\ARCHIVO\PLUGINS\VECTOR_MACHINE\STORAGE"

client = chromadb.PersistentClient(path=STORAGE_PATH)
print("Colecciones disponibles:", [c.name for c in client.list_collections()])

Colecciones disponibles: ['mr_analisis', 'me_mapas', 'me_mapas_mapas_sing', 'mr_corpus', 'ml_hilo_narrativo', 'me_turin', 'me_psicoanalisis', 'me_escano', 'ml_piezas_media', 'me_decoherencia', 'ml_recursos', 'mr_editoriales', 'ml_eventos', 'ml_personajes', 'mr_guiones', 'mr_poemas']


## Ingesta — colecciones de mod/restitutiva

Define aquí las colecciones a cargar. Edita `COLLECTIONS` y `LABELS`
según las colecciones que vayas a crear en el paso de ingesta.

In [8]:
# ── Editar aquí según colecciones de mod/restitutiva ────────────────────────
COLLECTIONS = [
     "mr_editoriales",
     "mr_analisis",
     "mr_corpus",
     "mr_poemas",
     "mr_guiones",
]

LABELS = {
     "mr_editoriales": "Editoriales",
     "mr_analisis":    "Análisis",
     "mr_corpus":      "Corpus",
     "mr_poemas":      "Poemas",
     "mr_guiones":     "Guiones",
}

# Paleta roja — mod/restitutiva
COLOR_MAP = {
    "Editoriales": "#c41e3a",
    "Análisis":    "#8b1428",
    "Corpus":      "#e87070",
    "Poemas":      "#f0a0a0",
    "Guiones":     "#a03050",
}

if not COLLECTIONS:
    print("⚠ COLLECTIONS vacío — descomenta las colecciones antes de ejecutar")
else:
    rows = []
    for col_name in COLLECTIONS:
        col = client.get_collection(col_name)
        result = col.get(include=["embeddings", "documents", "metadatas"])
        for doc_id, emb, doc, meta in zip(
            result["ids"], result["embeddings"], result["documents"], result["metadatas"]
        ):
            rows.append({
                "id":        doc_id,
                "coleccion": LABELS[col_name],
                "col_raw":   col_name,
                "bloque":    meta.get("bloque", "?"),
                "tipo":      meta.get("tipo", meta.get("subtipo", "?")),
                "marca":     meta.get("marca", doc_id),
                "texto":     doc[:120] + "…" if len(doc) > 120 else doc,
                "embedding": emb,
            })

    df = pd.DataFrame(rows)
    embeddings_matrix = np.array(df["embedding"].tolist())
    print(f"Total piezas cargadas: {len(df)}")
    print(df.groupby("coleccion").size().to_string())

Total piezas cargadas: 33
coleccion
Análisis       16
Corpus          8
Editoriales     4
Guiones         3
Poemas          2


## Proyección UMAP + visualizaciones

Las siguientes celdas son idénticas al cuaderno negro — solo cambia la paleta.

In [9]:
UMAP_PARAMS = dict(n_neighbors=5, min_dist=0.2, random_state=42, low_memory=False)

reducer_2d = umap.UMAP(n_components=2, **UMAP_PARAMS)
proj_2d = reducer_2d.fit_transform(embeddings_matrix)
df["x"] = proj_2d[:, 0]
df["y"] = proj_2d[:, 1]

reducer_3d = umap.UMAP(n_components=3, **UMAP_PARAMS)
proj_3d = reducer_3d.fit_transform(embeddings_matrix)
df["x3"] = proj_3d[:, 0]
df["y3"] = proj_3d[:, 1]
df["z3"] = proj_3d[:, 2]

print("Proyección 2D shape:", proj_2d.shape)
print("Proyección 3D shape:", proj_3d.shape)

c:\Users\aleph\OASIS\aleph-scriptorium\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Proyección 2D shape: (33, 2)
Proyección 3D shape: (33, 3)


c:\Users\aleph\OASIS\aleph-scriptorium\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [15]:
fig2d = px.scatter(
    df, x="x", y="y",
    color="coleccion",
    color_discrete_map=COLOR_MAP,
    hover_data={"marca": True, "tipo": True, "texto": True, "x": False, "y": False},
    text="marca",
    title="mod/restitutiva — Espacio de embeddings 2D (UMAP)",
    width=900, height=650,
)
fig2d.update_traces(textposition="top center", textfont_size=9, marker=dict(size=10, opacity=0.85))
fig2d.update_layout(
    legend_title_text="Colección",
    plot_bgcolor="#1a0a0a",
    paper_bgcolor="#1a0a0a",
    font_color="#e0d0d0",
)
show(fig2d)

In [11]:
fig3d = px.scatter_3d(
    df, x="x3", y="y3", z="z3",
    color="coleccion",
    color_discrete_map=COLOR_MAP,
    hover_data={"marca": True, "tipo": True, "texto": True,
                "x3": False, "y3": False, "z3": False},
    text="marca",
    title="mod/restitutiva — Espacio de embeddings 3D (UMAP)",
    width=900, height=700,
)
fig3d.update_traces(textposition="top center", textfont_size=8, marker=dict(size=5, opacity=0.9))
fig3d.update_layout(
    legend_title_text="Colección",
    paper_bgcolor="#1a0a0a",
    font_color="#e0d0d0",
    scene=dict(
        bgcolor="#1a0a0a",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    ),
)
show(fig3d)

In [12]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings_matrix)
    scores[k] = silhouette_score(embeddings_matrix, labels)

best_k = max(scores, key=scores.get)
print(f"K óptimo: {best_k} (score={scores[best_k]:.3f})")

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(embeddings_matrix).astype(str)

fig_cluster = px.scatter(
    df, x="x", y="y",
    color="cluster", symbol="coleccion",
    hover_data={"marca": True, "tipo": True, "texto": True, "coleccion": True, "x": False, "y": False},
    text="marca",
    title=f"Clusters semánticos (K={best_k}) — mod/restitutiva",
    width=900, height=650,
)
fig_cluster.update_traces(textposition="top center", textfont_size=8, marker=dict(size=10, opacity=0.85))
fig_cluster.update_layout(plot_bgcolor="#1a0a0a", paper_bgcolor="#1a0a0a", font_color="#e0d0d0")
show(fig_cluster)

print("\nComposición de clusters:")
print(df.groupby(["cluster", "coleccion"])["marca"].apply(list).to_string())

K óptimo: 2 (score=0.118)



Composición de clusters:
cluster  coleccion  
0        Análisis       [an-ed1-corriente, an-ed2-corriente, an-ed2-me...
         Corpus                    [co-corriente-linaje, co-taxonomia-3e]
         Editoriales    [Ed.1 Primero de Mayo, Ed.2 Materialismo, Ed.3...
         Poemas         [poema-hilo-rojo-2026-04-15, poema-solicitud-t...
1        Análisis       [an-ed1-taxonomia, an-ed1-mecanismos, an-ed1-e...
         Corpus         [co-exclusion, co-taxonomia-3a-3b, co-taxonomi...
         Guiones        [guion-materialismo-2024, guion-arte-2025, gui...


## Divergencia corpus.md ↔ geometría

In [13]:
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=4, metric="cosine").fit(embeddings_matrix)
distances, indices = nbrs.kneighbors(embeddings_matrix)

divergencias = []
for i, (dists_i, idxs_i) in enumerate(zip(distances, indices)):
    pieza = df.iloc[i]
    vecinos = df.iloc[idxs_i[1:]]
    vecinos_col = vecinos["col_raw"].value_counts()
    col_mayoritaria = vecinos_col.index[0]
    if col_mayoritaria != pieza["col_raw"] and vecinos_col.iloc[0] >= 2:
        divergencias.append({
            "marca":          pieza["marca"],
            "col_asignada":   pieza["coleccion"],
            "col_geometrica": LABELS[col_mayoritaria],
            "vecinos":        list(vecinos["marca"]),
            "dist_media":     round(float(dists_i[1:].mean()), 4),
        })

if divergencias:
    print(f"Piezas con divergencia corpus ↔ geometría ({len(divergencias)}):\n")
    for d in divergencias:
        print(f"  {d['marca']}")
        print(f"    Asignada:   {d['col_asignada']}")
        print(f"    Geometría:  {d['col_geometrica']}")
        print(f"    Vecinos:    {d['vecinos']}")
        print(f"    Dist media: {d['dist_media']}\n")
else:
    print("Sin divergencias — la taxonomía del Archivero coincide con la geometría.")

Piezas con divergencia corpus ↔ geometría (14):

  Ed.1 Primero de Mayo
    Asignada:   Editoriales
    Geometría:  Análisis
    Vecinos:    ['an-ed1-taxonomia', 'an-ed1-corriente', 'co-corriente-linaje']
    Dist media: 0.2922

  Ed.2 Materialismo
    Asignada:   Editoriales
    Geometría:  Análisis
    Vecinos:    ['an-ed2-corriente', 'an-ed2-emergencias', 'an-ed1-corriente']
    Dist media: 0.3144

  Ed.3 Arte/Estética
    Asignada:   Editoriales
    Geometría:  Análisis
    Vecinos:    ['an-ed3-taxonomia', 'poema-hilo-rojo-2026-04-15', 'an-ed1-corriente']
    Dist media: 0.3116

  Ed.4 Guerra+Capital
    Asignada:   Editoriales
    Geometría:  Análisis
    Vecinos:    ['an-ed4-taxonomia', 'co-taxonomia-3e', 'an-ed4-corriente']
    Dist media: 0.1792

  an-ed1-taxonomia
    Asignada:   Análisis
    Geometría:  Corpus
    Vecinos:    ['co-taxonomia-3a-3b', 'co-taxonomia-3c-3d', 'co-exclusion']
    Dist media: 0.2194

  an-ed1-emergencias
    Asignada:   Análisis
    Geometría:  Corpu

## Ejemplo de query semántica

In [18]:
# ── Query semántica sobre las colecciones de mod/restitutiva ─────────────────
# Edita QUERY y COLECCIONES_QUERY para explorar el espacio vectorial.

QUERY = "¿Línea editorial general?"
COLECCIONES_QUERY = ["mr_analisis", "mr_corpus", "mr_editoriales"]
N_RESULTADOS = 3

print(f"Query: «{QUERY}»\n")
print("=" * 70)

for col_name in COLECCIONES_QUERY:
    col = client.get_collection(col_name)
    result = col.query(
        query_texts=[QUERY],
        n_results=N_RESULTADOS,
        include=["documents", "metadatas", "distances"],
    )
    docs      = result["documents"][0]
    metas     = result["metadatas"][0]
    distances = result["distances"][0]

    print(f"\n▶ {LABELS[col_name]}")
    print("-" * 50)
    for doc, meta, dist in zip(docs, metas, distances):
        etiqueta = meta.get("marca", meta.get("editorial", meta.get("seccion", "?")))
        print(f"  [{dist:.3f}]  {etiqueta}")
        print(f"           {doc[:120]}…")


Query: «¿Línea editorial general?»


▶ Análisis
--------------------------------------------------
  [1.106]  primero-mayo-2024
           Análisis Bartleby Ed.1 — Emergencias y Ausencias. Editorial Primero de Mayo 2024.

Emergencias: E.01 El problema de la t…
  [1.151]  arte-2025
           Análisis Bartleby Ed.3 — Taxonomía. Editorial Arte/Estética, 2025.

Nueva rama: ESTÉTICA MARXISTA subordinada a HEGEMONÍ…
  [1.164]  primero-mayo-2024
           Análisis Bartleby Ed.1 — Taxonomía funcional. Editorial Primero de Mayo 2024.

Árbol funcional (7 nodos, todos con verbo…

▶ Corpus
--------------------------------------------------
  [1.167]  mecanismos_retóricos
           Corpus BARTLEBY — Mecanismos retóricos (frecuencias acumuladas en n=4).

Mecanismos del nick restitutiva con frecuencia …
  [1.205]  taxonomia_estetica_metodo
           Corpus BARTLEBY — Taxonomía funcional (partes 3c y 3d): Registro estético y método.

3c. Registro estético (arte-2025): …
  [1.255]  taxonomia_instit

## Exportar a GH Pages — `docs/restitutiva/cuadernos/`

In [19]:
import os
import yaml
from datetime import date

EXPORT_DIR  = r"C:\Users\aleph\OASIS\aleph-scriptorium\DocumentMachineSDK\docs\restitutiva\cuadernos"
DATA_FILE   = r"C:\Users\aleph\OASIS\aleph-scriptorium\DocumentMachineSDK\docs\_data\cuadernos.yml"
os.makedirs(EXPORT_DIR, exist_ok=True)

exports = {
    "corpus_2d": {
        "fig":         fig2d,
        "title":       "Espacio de embeddings 2D",
        "descripcion": "Proyección UMAP 2D de las 5 colecciones (33 fragmentos). Paleta roja — corriente restitutiva.",
    },
    "corpus_3d": {
        "fig":         fig3d,
        "title":       "Espacio de embeddings 3D",
        "descripcion": "Proyección UMAP 3D rotable. Permite ver la separación entre clusters de editoriales, análisis y corpus.",
    },
    "corpus_clusters": {
        "fig":         fig_cluster,
        "title":       f"Clusters semánticos (K={best_k})",
        "descripcion": f"Agrupación K-means (K={best_k}) sobre el espacio vectorial. Revela si la taxonomía del Archivero coincide con la geometría.",
    },
}

# ── Exportar HTML ─────────────────────────────────────────────────────────────
for cuaderno_id, info in exports.items():
    fname = f"{cuaderno_id}.html"
    dest  = os.path.join(EXPORT_DIR, fname)
    info["fig"].write_html(dest, include_plotlyjs="cdn", full_html=True)
    print(f"✓ {fname}  ({os.path.getsize(dest) // 1024} KB)")

# ── Actualizar _data/cuadernos.yml ────────────────────────────────────────────
colecciones_str = " · ".join(LABELS.values())
hoy = date.today().isoformat()

registros = []
for cuaderno_id, info in exports.items():
    registros.append({
        "id":          cuaderno_id,
        "title":       info["title"],
        "file":        f"restitutiva/cuadernos/{cuaderno_id}.html",
        "descripcion": info["descripcion"],
        "fecha":       hoy,
        "colecciones": colecciones_str,
    })

header = (
    "# Cuadernos vectoriales — mod/restitutiva\n"
    "# Generado/actualizado automáticamente por la celda de exportación del notebook.\n"
    "# Cada entrada se convierte en una card en el catálogo.\n\n"
)
with open(DATA_FILE, "w", encoding="utf-8") as f:
    f.write(header)
    yaml.dump(registros, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f"\n✓ {DATA_FILE} actualizado ({len(registros)} cuadernos)")


✓ corpus_2d.html  (16 KB)
✓ corpus_3d.html  (17 KB)
✓ corpus_clusters.html  (18 KB)
